# Week 7 group project: Object Recognition
AI for Good - Minor ADSAI

[Open this notebook in Google Colab](https://colab.research.google.com/github/BTuyn/AI4G/blob/main/Week%207/Week7_Starter_Object_Recognition.ipynb)

**Your group's mission:** build a very simple app (an MVP) that does
**object recognition: finding objects in a photo, with a box and a label** on photos you take yourselves, in or outside the school, find out how
good it is, and teach the rest of the class about it.

*Question for your group: what is in the photo, where is it, and how many are there?*

**Timeline**

| Time | What |
|------|------|
| 10:45-11:30 | **Phase 1 (45 min):** take photos and come up with your MVP - no laptops needed (read sections 1 and 3 for ideas) |
| 11:30 | **Stand-up:** 3 minutes per group - your MVP idea, your photos, who does what |
| 12:00-14:30 | **Phase 2:** photos into Colab, build the MVP and make it work, ground truth, evaluation, presentation (sections 0-8) |
| 14:45-15:30 | **Presentations:** your MVP, 10 minutes per group + 3 minutes questions |

**Your presentation must explain**

1. **What the technique is** - what goes in, what comes out, what it is used for (section 1)
2. **What data processing is needed** - from a phone photo to something the model can use, and the ground truth (sections 4 and 6)
3. **What you did** - your idea, your photos, how you built your MVP (sections 3 and 5)
4. **How you evaluate it** - which metric, why, and your result (section 7)
5. **The end result: your MVP** - show it working, and where it fails (section 8)

**Organise your group.** You are a big group, so split the work. A split
that works: a **photo team** (collects photos, makes the ground truth), a
**build team** (runs this notebook, builds the MVP), an **evaluation team**
(chooses and calculates the metric), and a **presentation team** (slides,
demo, timing). Pick one person who keeps an eye on the clock. Share this
notebook through Google Drive so everybody can see it.

## My project: how many bikes are in the rack?

**Felix Reichle.** I missed the workshop, so I did this project on my own.

**Question:** how many bikes are in a bike rack right now?
**User:** a student who wants to know whether there is still a free spot before walking to the rack.


## 0. Setup - run these cells once

The first cell is only needed for iPhone photos in HEIC format (or set the
iPhone camera to *Settings -> Camera -> Formats -> Most Compatible*). The
second cell loads the tools and some helpers:

- `upload_to("photos")` - choose photos on your laptop; they land in the folder `photos`
- `take_photo("photos", "name.jpg")` - take a photo with the laptop camera
- `load_folder("photos")` - load all photos of a folder: turned the right way up and made smaller
- `show([...])` - show pictures next to each other
- `save_result(name)` - save the picture you just made in the folder `results`, for your slides
- `download_results()` - download the folder `results` as a zip

In [ ]:
# Only needed for iPhone HEIC photos
!pip install -q pillow-heif

In [ ]:
# Run, do not read
import warnings
warnings.filterwarnings("ignore")
import os, glob, base64, shutil, urllib.request, zipfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageOps
import torch
import torchvision.transforms.functional as TF
torch.set_grad_enabled(False)
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except ImportError:
    pass
os.makedirs("results", exist_ok=True)

def show(images, titles=None, size=3.5):
    if not isinstance(images, list):
        images = [images]
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    if len(images) == 1:
        axes = [axes]
    for k in range(len(images)):
        axes[k].imshow(images[k], cmap="gray" if images[k].ndim == 2 else None)
        axes[k].axis("off")
        if titles:
            axes[k].set_title(titles[k], fontsize=9)
    plt.show()

def save_result(name):
    # call this INSIDE the same cell, right after a picture is drawn
    plt.savefig(os.path.join("results", name), dpi=120, bbox_inches="tight")
    print("saved results/" + name)

def download_results():
    from google.colab import files
    shutil.make_archive("results", "zip", "results")
    files.download("results.zip")

def upload_to(folder):
    from google.colab import files
    os.makedirs(folder, exist_ok=True)
    for name, content in files.upload().items():
        with open(os.path.join(folder, name), "wb") as f:
            f.write(content)

def take_photo(folder, name):
    from IPython.display import display, Javascript
    from google.colab.output import eval_js
    os.makedirs(folder, exist_ok=True)
    display(Javascript('''
    async function takePhoto() {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = 'Capture';
      div.appendChild(button);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => button.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', 0.9);
    }'''))
    data_url = eval_js("takePhoto()")
    with open(os.path.join(folder, name), "wb") as f:
        f.write(base64.b64decode(data_url.split(",")[1]))
    print("saved", os.path.join(folder, name))

def load_folder(folder, max_size=1024):
    photos = {}
    for f in sorted(glob.glob(os.path.join(folder, "*"))):
        if "_mask" in os.path.basename(f):
            continue
        try:
            im = ImageOps.exif_transpose(Image.open(f)).convert("RGB")
        except Exception:
            print("skipped (not a photo):", f)
            continue
        im.thumbnail((max_size, max_size))
        photos[os.path.basename(f)] = np.array(im)
    print(folder, ":", len(photos), "photos")
    return photos

def download_examples(folder, urls):
    os.makedirs(folder, exist_ok=True)
    for url in urls:
        target = os.path.join(folder, url.split("/")[-1])
        if not os.path.exists(target):
            urllib.request.urlretrieve(url, target)

print("Ready")

## 1. What is object recognition?

- **Classification** gives one label for the whole photo: "this is a bicycle".
- **Object detection** finds *every* object, with a **box**, a **label** and a
  **confidence** (how sure the model is, from 0 to 1). Because you know where
  each object is, you can **count**.

The starter offers Mask R-CNN and OWLv2. I built my MVP before I had this starter, with
**Faster R-CNN** (ResNet-50 backbone, version 2, pre-trained on COCO), and I kept it:
I only need boxes, not the masks of Mask R-CNN, and COCO already knows `bicycle`, so I do not need OWLv2's free text search either.

The **threshold** decides which boxes you keep: low = more found, more
mistakes; high = surer, but more missed.


In [ ]:
# My detector: Faster R-CNN instead of the starter's two models
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
coco_weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
coco_model = fasterrcnn_resnet50_fpn_v2(weights=coco_weights).eval()
COCO = [name for name in coco_weights.meta["categories"]]

def find(picture, what, threshold=0.5, draw=True, title=""):
    boxes = []                                    # (label, score, [x1, y1, x2, y2])
    out = coco_model([TF.to_tensor(picture)])[0]
    for box, lab, score in zip(out["boxes"], out["labels"], out["scores"]):
        if COCO[int(lab)] in what and score >= threshold:
            boxes.append((COCO[int(lab)], float(score), box.tolist()))
    if draw:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.imshow(picture)
        for label, score, (x1, y1, x2, y2) in boxes:
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="yellow", linewidth=2))
            ax.text(x1, y1, label + " " + str(round(score, 2)), fontsize=7, backgroundcolor="yellow")
        ax.axis("off")
        ax.set_title(title)
    return boxes

def counts(boxes, what):
    result = {}
    for w in what:
        result[w] = sum(1 for b in boxes if b[0] == w)
    return result

print("COCO knows:", [n for n in COCO if n not in ("__background__", "N/A")])


## 2. Try it now

One example photo. This is already a tiny MVP: a photo in, boxes and counts out.


In [ ]:
street = np.array(Image.open(urllib.request.urlopen("http://images.cocodataset.org/val2017/000000174482.jpg")).convert("RGB"))
boxes = find(street, ["car", "bicycle"], 0.5, title="Faster R-CNN")
plt.show()
print(counts(boxes, ["car", "bicycle"]))


## 3. Your photos

**Ideas** (or think of your own):

- *Free seats:* count chairs and people in the study area or the canteen
- *Bike shed:* how many bicycles are parked, and is it full?
- *Litter patrol:* count bottles, cans or cups left on the tables or outside
- *Green campus:* how many trees and plants are around the building?
- *Lending desk:* how many laptops are in use in a classroom?

Decide **what** your app counts, and **for whom**. Then go and take photos.

**Rules for taking photos**

- **People only with permission.** Only photograph people who say yes - accept
  a no. Other people in the school must not be recognisable: aim away, or
  photograph them from behind.
- **Never put photos of people online.** Not in your portfolio (it is public)
  and not in a shared folder outside the group. Before you upload this notebook
  anywhere: *Edit -> Clear all outputs*. Delete the photos after this week.
- **Make it hard for the model.** Different places, light, distances and
  angles, and a few difficult photos (crowded, dark, half hidden). A test with
  only easy photos tells you nothing.
- **Enough photos:** at least 10, more is better. Any phone photo works.

**Getting your photos into Colab:** `upload_to("photos")` from the laptop
(send them from your phone by AirDrop, e-mail or Google Drive first), drag a
zip into the file panel on the left and run `!unzip -q yourfile.zip -d photos`,
or use the laptop camera with `take_photo("photos", "name.jpg")`.

**What I did:** my app counts bikes (the *bike shed* idea). I took 9 photos with my phone, from nearly empty racks to packed ones: outdoors, indoors, in the evening and at night.
I blurred the people and a number plate in the background and removed the GPS data, so the folder `photos` holds these copies.


In [ ]:
folder = "photos"
what = ["bicycle"]                              # the object my app finds

if not os.path.isdir(folder):                   # in Colab: upload the photos first
    upload_to(folder)

photos = load_folder(folder)
show([photos[n] for n in photos][:4], list(photos)[:4], size=3)


## 4. Data processing

A phone photo is not ready for a model. Run the cell: it shows what happens to
one of your photos on its way in. **Use this for part 2 of your
presentation.**

The files in `photos` are my blurred copies at 1600 x 1200. The originals from my phone were 4096 x 3072, so step 1 below starts from the copy.

In [ ]:
# Run, do not read: what happens to a phone photo before the model sees it
name = list(photos)[0]
original = Image.open(glob.glob(os.path.join(folder, name))[0])
print("1. the file from the phone:     ", original.size[0], "x", original.size[1], "pixels,", original.mode)
upright = ImageOps.exif_transpose(original)
print("2. turned the right way up:     ", "yes" if upright.size != original.size else "no rotation needed",
      "(phones store the rotation separately, in the EXIF data)")
print("3. made smaller (max 1024):     ", photos[name].shape[1], "x", photos[name].shape[0], "pixels")
print("4. a grid of numbers:           ", photos[name].shape, "=", photos[name].size, "numbers from 0 to 255")
x = TF.to_tensor(photos[name])
print("5. scaled to 0-1 for the model: ", round(float(x.min()), 2), "to", round(float(x.max()), 2))
show([np.array(original.convert("RGB")), photos[name]], ["from the phone", "what we use"])

**For object recognition, the ground truth is a count per photo** (section 6):
you count the objects yourself, before you look at the model. For a full
evaluation you also judge every box the model draws: right or wrong.

## 5. My MVP

A photo in, an answer out: count the `bicycle` boxes above the threshold.
I started at 0.6, but that counted far too many bikes on the full racks. The threshold plot in section 7 shows 0.85 works better, so I use that.


In [ ]:
threshold = 0.85

def my_app(picture, name=""):
    boxes = find(picture, what, threshold, title=name)
    c = counts(boxes, what)
    return "How many bikes are in the rack? -> " + str(c["bicycle"])

for name in photos:
    answer = my_app(photos[name], name)
    plt.show()
    print(name, "->", answer)


## 6. Ground truth

I counted the bikes on every original photo myself **before** looking at the boxes (`truth`). I counted every bike in the foreground, also the ones that are only partly visible.
When I checked again later I changed one count from 6 to 7 (the photo with the black gate).

Then I went through the boxes at threshold 0.85 and counted how many were right (`correct`). A box is wrong if it shows no bike, is a second box on a bike that already has one, or shows a bike in the background.
On the full racks I could not always tell which box was wrong. If a photo has more boxes than bikes, the extra boxes still count as wrong.


In [ ]:
truth = {      # how many there really are - counted by me
    "IMG20261101112208.jpg": {"bicycle": 5},
    "IMG20261104115853.jpg": {"bicycle": 7},
    "IMG20261104115900.jpg": {"bicycle": 7},
    "IMG20261105124253.jpg": {"bicycle": 9},
    "IMG20261105191915.jpg": {"bicycle": 13},
    "IMG20261105192223.jpg": {"bicycle": 10},
    "IMG20261105192349.jpg": {"bicycle": 12},
    "IMG20261105192358.jpg": {"bicycle": 10},
    "IMG20261105192406.jpg": {"bicycle": 10},
}
correct = {    # how many of the model's boxes were right, at threshold 0.85 - judged by me
    "IMG20261101112208.jpg": {"bicycle": 5},
    "IMG20261104115853.jpg": {"bicycle": 6},
    "IMG20261104115900.jpg": {"bicycle": 5},
    "IMG20261105124253.jpg": {"bicycle": 9},
    "IMG20261105191915.jpg": {"bicycle": 6},
    "IMG20261105192223.jpg": {"bicycle": 8},
    "IMG20261105192349.jpg": {"bicycle": 12},
    "IMG20261105192358.jpg": {"bicycle": 8},
    "IMG20261105192406.jpg": {"bicycle": 10},
}


## 7. Evaluate

Three numbers per kind of object:

- **Precision** = right boxes / all boxes the model drew - *how many false alarms?*
- **Recall** = right boxes / objects that are really there - *how many missed?*
- **Count error** = on average, how many off per photo - *what the user notices*

**Count error is my main metric:** my user only sees the number, so what matters is how far that number is off.

The second cell tries other thresholds: which one gives the smallest count error?


In [ ]:
# Run, do not read
found = {}
for name in truth:
    found[name] = find(photos[name], what, 0.0, draw=False)   # every box, any confidence

def evaluate(threshold, verbose=True):
    total = {}
    for w in what:
        drawn = sum(sum(1 for b in found[n] if b[0] == w and b[1] >= threshold) for n in truth)
        real = sum(truth[n][w] for n in truth)
        errors = [abs(sum(1 for b in found[n] if b[0] == w and b[1] >= threshold) - truth[n][w]) for n in truth]
        total[w] = float(np.mean(errors))
        if verbose:
            right = sum(correct[n][w] for n in truth)
            print(w.ljust(12), "boxes", drawn, " right", right, " really there", real,
                  " precision", round(right / drawn, 2) if drawn else "-",
                  " recall", round(right / real, 2) if real else "-",
                  " count error", round(total[w], 2))
    return total

print("At threshold", threshold, "(precision and recall use your judgements in 'correct'):")
evaluate(threshold);

In [ ]:
thresholds = [round(t, 2) for t in np.arange(0.1, 1.0, 0.05)]
errors = [evaluate(t, verbose=False)["bicycle"] for t in thresholds]
for t, e in zip(thresholds, errors):
    print("threshold", t, " count error per photo:", round(e, 2))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(thresholds, errors, marker="o")
ax.axvline(threshold, color="grey", ls="--", label="threshold used: " + str(threshold))
ax.set(xlabel="threshold (minimum confidence)", ylabel="count error per photo", title="Count error per threshold")
ax.legend()
save_result("threshold.png")
plt.show()

# 0.85 and 0.90 have the same error, so which way is each one off?
for t in [0.85, 0.9]:
    drawn = sum(sum(1 for b in found[n] if b[1] >= t) for n in truth)
    print("threshold", t, ":", drawn, "boxes for", sum(truth[n]["bicycle"] for n in truth), "bikes")


**Questions for your presentation:** Which threshold is best for your app,
and which mistake does it make more: false alarms or misses? Which photos went
wrong, and why (light, distance, hidden, look-alikes)? Is the app good enough
for your user?

### My evaluation conclusions

At threshold 0.85 the count is off by **1.33 bikes per photo** on average, against 5.56 at the 0.6 I started with. Only 3 of the 9 counts are exact, and precision and recall are both 0.83. False alarms and misses are in balance: 14 of each. 0.90 has the same count error but undercounts, which could send someone to a full rack, so I chose 0.85.
For a student that is good enough on a quiet rack, but not to decide whether the last spot is still free.

**Why the worst photos fail:** on full racks the bikes overlap, so one threshold cannot fix everything. Lower thresholds give several boxes per bike, higher ones lose real bikes. The night photo is the worst: 8 boxes for 13 bikes. A correct count can also hide mistakes: one indoor photo has 10 boxes for 10 bikes, but 2 of those boxes are wrong.

**Limits:** I chose the threshold on the same 9 photos I report the results on, so the real error is probably higher. The MVP also counts bikes, not free spots: for that it still needs to know how many bikes fit in the rack.


## 8. End result and presentation checklist

Make the pictures for your slides with `save_result("name.png")` (in the same
cell as the picture) and download them all with `download_results()`.

| Part of your presentation | Where in this notebook | Done? |
|---------------------------|------------------------|-------|
| 1. What is the technique? input, output, what is it for | section 1 | yes |
| 2. Which data processing is needed? photo -> model, and the ground truth | sections 4 and 6 | yes |
| 3. What did you do? your idea, your photos, how you built your MVP | sections 3 and 5 | yes |
| 4. How do you evaluate it? metric, why, your result | section 7 | yes |
| 5. The end result: your MVP working, and where it fails | section 8 | yes |
| Extra: which photo is your best example, and which your worst? | below | yes |

Also prepare an answer to: **who could this app fail, and who could be harmed
by it?** The class will ask.

In [ ]:
# The end result: your app on your best and your worst photo - saved for the slides
best, worst = "IMG20261101112208.jpg", "IMG20261105191915.jpg"   # sparse rack in daylight, dense rack at night
for name, label in [(best, "best"), (worst, "worst")]:
    print(my_app(photos[name], label + ": " + name))
    save_result(label + ".png")
    plt.show()
# download_results()

### Who could this app fail, and who could be harmed by it?

- **Students at busy racks or after dark:** it is weakest exactly when they need it most. The harm is a wasted walk.
- **Owners of unusual bikes:** I did not test cargo bikes or adapted bikes, so they may not be counted.
- **People walking past:** a camera aimed at a rack also films them. That is why I blurred people and a number plate in my photos.
- **Bike owners, if it is misused:** with an error this size, the count should not decide things like removing bikes.
